For this demo, we will use the [MIT Restaurant Corpus](https://groups.csail.mit.edu/sls/downloads/restaurant/) -- a dataset of transcriptions of spoken utterances about restaurants.

The dataset has following entity types:

* 'B-Rating'
* 'I-Rating',
* 'B-Amenity',
* 'I-Amenity',
* 'B-Location',
* 'I-Location',
* 'B-Restaurant_Name',
* 'I-Restaurant_Name',
* 'B-Price',
* 'B-Hours',
* 'I-Hours',
* 'B-Dish',
* 'I-Dish',
* 'B-Cuisine',
* 'I-Price',
* 'I-Cuisine'

Let us load the dataset and see what are we working with.

In [6]:
import pandas as pd
import numpy as np

In [2]:
sent_train = input('Path to sent_train')
label_train = input('Path to label_train')
sent_test = input('Path to sent_test')
label_test = input('Path to label_test')

In [3]:
with open(sent_train, 'r') as train_sent_file:
  train_sentences = train_sent_file.readlines()

with open(label_train, 'r') as train_labels_file:
  train_labels = train_labels_file.readlines()

with open(sent_test, 'r') as test_sent_file:
  test_sentences = test_sent_file.readlines()

with open(label_test, 'r') as test_labels_file:
  test_labels = test_labels_file.readlines()


In [7]:
train_set = pd.DataFrame({'sentences': train_sentences, 'labels': train_labels})
train_set

,sentences,labels
0,2 start restaurants with inside dining \n,B-Rating I-Rating O O B-Amenity I-Amenity \n
1,34 \n,O \n
2,5 star resturants in my town \n,B-Rating I-Rating O B-Location I-Location I-Lo...
3,98 hong kong restaurant reasonable prices \n,O B-Restaurant_Name I-Restaurant_Name O B-Pric...
4,a great lunch spot but open till 2 a m passims...,O O O O O B-Hours I-Hours I-Hours I-Hours I-Ho...
...,...,...
7655,yes please locate the nearest seafood restaura...,O O O O B-Location B-Cuisine O \n
7656,yes we are looking for a formal restaurant tod...,O O O O O O B-Amenity O O O O O O O O O B-Amen...
7657,yes we need a to stop at five guys for a nice ...,O O O O O O O B-Restaurant_Name I-Restaurant_N...
7658,yes we need to find a cheap deli with good hou...,O O O O O O B-Price B-Cuisine O B-Rating B-Hou...


In [9]:
test_set = pd.DataFrame({'sentences': test_sentences, 'labels': test_labels})
test_set

,sentences,labels
0,a four star restaurant with a bar \n,O B-Rating I-Rating O B-Location I-Location B-...
1,any asian cuisine around \n,O B-Cuisine O B-Location \n
2,any bbq places open before 5 nearby \n,O B-Cuisine O B-Hours I-Hours I-Hours B-Locati...
3,any dancing establishments with reasonable pri...,O B-Location I-Location O B-Price O \n
4,any good cheap german restaurants nearby \n,O O B-Price B-Cuisine O B-Location \n
...,...,...
1516,will waffle house accept a prepaid visa gift c...,O B-Restaurant_Name I-Restaurant_Name O O B-Am...
1517,yes please get me mcdonalds phone number in pa...,O O O O B-Restaurant_Name O O O B-Location I-L...
1518,yes the new diner on south street please \n,O O O B-Cuisine O B-Location I-Location O \n
1519,yes we need some chicken for our new diet so c...,O O O O B-Dish O O O O O B-Restaurant_Name I-R...


Let us see some example data points.

In [11]:
# Print the 6th sentence in the test set i.e. index value 5.
print(test_set.loc[5, 'sentences'])

# Print the labels of this sentence
print(test_set.loc[5, 'labels'])

any good ice cream parlors around 

O B-Rating B-Cuisine I-Cuisine I-Cuisine B-Location 



#Defining Features for Custom NER

First, let us install the required modules.

In [ ]:
# Install pycrf and crfsuit packages using pip command
# !pip install pycrf

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pycrf: filename=pycrf-0.0.1-py3-none-any.whl size=1907 sha256=22b7feaa6b5728af1e48b22e8a7f4aeb0feb25edc1708bf27ae8b3df9c4ef847
  Stored in directory: c:\users\arnig\appdata\local\pip\cache\wheels\9e\f7\ef\9cfb9c92784d1ed9253ab1b70c18de241061871de27e547a37
Successfully built pycrf


In [ ]:
# !pip install sklearn-crfsuite

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)




We will now start with computing features for our input sequences.

We have defined the following features for CRF model building:

- f1 = input word is in lower case; (identity function)
- f2 = last 3 characters of word; (suffix function for present continuous)
- f3 = last 2 characers of word; (suffix function for past tense)
- f4 = 1; if the word is in uppercase, 0 otherwise;
- f5 = 1; if word is a number; otherwise, 0 
- f6= 1; if the word starts with a capital letter; otherwise, 0


In [25]:
#Define a function to get the above defined features for a word.
def getFeaturesForOneWord(sentence, pos):
  word = sentence.split()[pos]
  
  # Creates a list of strings
  features = [
    'word.lower=' + word.lower(), # serves as word id 
    'word[-3:]=' + word[-3:],     # last three characters
    'word[-2:]=' + word[-2:],     # last two characters
    'word.isupper=%s' % word.isupper(),  # is the word in all uppercase
    'word.isdigit=%s' % word.isdigit(),  # is the word a number
    'words.startsWithCapital=%s' % word[0].isupper() # is the word starting with a capital letter
  ]
 
  if(pos > 0):
    prev_word = sentence[pos-1]
    features.extend([
    'prev_word.lower=' + prev_word.lower(), 
    'prev_word.isupper=%s' % prev_word.isupper(),
    'prev_word.isdigit=%s' % prev_word.isdigit(),
    'prev_words.startsWithCapital=%s' % prev_word[0].isupper()
  ])
  else:
    features.append('BEG') # feature to track begin of sentence 
 
  if(pos == len(sentence.split())-1):
    features.append('END') # feature to track end of sentence
 
  return features

#Computing Features 

Define a function to get features for a sentence using the already defined 'getFeaturesForOneWord' function

In [31]:
# Define a function to get features for a sentence 
# using the 'getFeaturesForOneWord' function.
def getFeaturesForOneSentence(sentence):
    positions = range(len(sentence.split()))
    return [getFeaturesForOneWord(sentence, pos) for pos in positions]

Define function to get the labels for a sentence.

In [32]:
# Define a function to get the labels for a sentence.
def getLabelsForOneSentence(labels):
    return labels.split()

Example features for a sentence


In [34]:
# Apply function 'getFeaturesForOneSentence' to get features on a single sentence which is at index value 5 in train_sentences
print(train_set.loc[5, 'sentences'])
getFeaturesForOneSentence(train_set.loc[5, 'sentences'])

a place that serves soft serve ice cream 



[['word.lower=a',
  'word[-3:]=a',
  'word[-2:]=a',
  'word.isupper=False',
  'word.isdigit=False',
  'words.startsWithCapital=False',
  'BEG'],
 ['word.lower=place',
  'word[-3:]=ace',
  'word[-2:]=ce',
  'word.isupper=False',
  'word.isdigit=False',
  'words.startsWithCapital=False',
  'prev_word.lower=a',
  'prev_word.isupper=False',
  'prev_word.isdigit=False',
  'prev_words.startsWithCapital=False'],
 ['word.lower=that',
  'word[-3:]=hat',
  'word[-2:]=at',
  'word.isupper=False',
  'word.isdigit=False',
  'words.startsWithCapital=False',
  'prev_word.lower= ',
  'prev_word.isupper=False',
  'prev_word.isdigit=False',
  'prev_words.startsWithCapital=False'],
 ['word.lower=serves',
  'word[-3:]=ves',
  'word[-2:]=es',
  'word.isupper=False',
  'word.isdigit=False',
  'words.startsWithCapital=False',
  'prev_word.lower=p',
  'prev_word.isupper=False',
  'prev_word.isdigit=False',
  'prev_words.startsWithCapital=False'],
 ['word.lower=soft',
  'word[-3:]=oft',
  'word[-2:]=ft',
  'wo

In [36]:
getLabelsForOneSentence(train_set.loc[5, 'labels'])

['O', 'O', 'O', 'O', 'B-Dish', 'I-Dish', 'I-Dish', 'I-Dish']

Get the features for sentences of X_train and X_test and get the labels of Y_train and Y_test data.

In [42]:
X_train = [getFeaturesForOneSentence(sent) for sent in train_set.loc[:, 'sentences']]
y_train = [getLabelsForOneSentence(sent) for sent in train_set.loc[:, 'labels']]

X_test = [getFeaturesForOneSentence(sent) for sent in test_set.loc[:, 'sentences']]
y_test = [getLabelsForOneSentence(sent) for sent in test_set.loc[:, 'labels']]

In [68]:
y_test

[['O', 'B-Rating', 'I-Rating', 'O', 'B-Location', 'I-Location', 'B-Amenity'],
 ['O', 'B-Cuisine', 'O', 'B-Location'],
 ['O', 'B-Cuisine', 'O', 'B-Hours', 'I-Hours', 'I-Hours', 'B-Location'],
 ['O', 'B-Location', 'I-Location', 'O', 'B-Price', 'O'],
 ['O', 'O', 'B-Price', 'B-Cuisine', 'O', 'B-Location'],
 ['O', 'B-Rating', 'B-Cuisine', 'I-Cuisine', 'I-Cuisine', 'B-Location'],
 ['O', 'B-Rating', 'O', 'O', 'O', 'O', 'B-Dish', 'O', 'O', 'B-Price', 'O'],
 ['O', 'O', 'B-Cuisine', 'O', 'B-Location'],
 ['O', 'B-Cuisine', 'O', 'O', 'O', 'B-Dish', 'B-Amenity', 'I-Amenity'],
 ['O',
  'O',
  'B-Location',
  'I-Location',
  'I-Location',
  'O',
  'O',
  'B-Rating',
  'B-Dish',
  'O',
  'O',
  'O',
  'O',
  'B-Dish'],
 ['O',
  'O',
  'B-Location',
  'I-Location',
  'O',
  'O',
  'O',
  'B-Amenity',
  'I-Amenity'],
 ['O',
  'B-Price',
  'O',
  'B-Cuisine',
  'O',
  'B-Location',
  'I-Location',
  'I-Location',
  'I-Location'],
 ['O', 'O', 'B-Hours', 'I-Hours', 'I-Hours'],
 ['O', 'O', 'O', 'O', 'O', 'B

#CRF Model Training

 Now we have all the information we need to train our CRF. Let us see how we can do that.

In [35]:
import sklearn_crfsuite

from sklearn_crfsuite import metrics

We create a CRF object and passtraining data to it. The model then "trains" and learns the weights for feature functions.

In [43]:
# Build the CRF model.
crf = sklearn_crfsuite.CRF(max_iterations= 100)
crf.fit(X_train, y_train)

CRF(max_iterations=100)

#Model Testing and Evaluation 
The model is trained, let us now see how good it performs on the test data.

In [59]:
# Predicted values
predict_set = pd.DataFrame({'sentences':test_set.loc[:, 'sentences'], 'labels':crf.predict(X_test)})
predict_set

,sentences,labels
0,a four star restaurant with a bar \n,"[O, B-Rating, I-Rating, O, O, O, B-Amenity]"
1,any asian cuisine around \n,"[O, B-Cuisine, O, B-Location]"
2,any bbq places open before 5 nearby \n,"[O, B-Cuisine, O, B-Hours, I-Hours, I-Hours, B..."
3,any dancing establishments with reasonable pri...,"[O, O, O, O, B-Price, O]"
4,any good cheap german restaurants nearby \n,"[O, B-Rating, B-Price, B-Cuisine, O, B-Location]"
...,...,...
1516,will waffle house accept a prepaid visa gift c...,"[O, B-Cuisine, I-Cuisine, O, O, B-Amenity, I-A..."
1517,yes please get me mcdonalds phone number in pa...,"[O, O, O, O, B-Restaurant_Name, O, O, B-Locati..."
1518,yes the new diner on south street please \n,"[O, O, B-Restaurant_Name, I-Restaurant_Name, O..."
1519,yes we need some chicken for our new diet so c...,"[O, O, O, O, B-Dish, O, O, B-Amenity, I-Amenit..."


In [75]:
test_set.loc[:, 'labels'].apply(lambda x: x.split())

0       [O, B-Rating, I-Rating, O, B-Location, I-Locat...
1                           [O, B-Cuisine, O, B-Location]
2       [O, B-Cuisine, O, B-Hours, I-Hours, I-Hours, B...
3              [O, B-Location, I-Location, O, B-Price, O]
4               [O, O, B-Price, B-Cuisine, O, B-Location]
                              ...                        
1516    [O, B-Restaurant_Name, I-Restaurant_Name, O, O...
1517    [O, O, O, O, B-Restaurant_Name, O, O, O, B-Loc...
1518    [O, O, O, B-Cuisine, O, B-Location, I-Location...
1519    [O, O, O, O, B-Dish, O, O, O, O, O, B-Restaura...
1520                   [O, O, O, O, O, O, B-Dish, I-Dish]
Name: labels, Length: 1521, dtype: object

In [78]:
# Calculate the f1 score using the test data
print(crf.score(X_test, y_test))
print(metrics.flat_f1_score(y_test, predict_set.loc[:, 'labels'], average='weighted'))

0.8554994388327721
0.8532527417164291


In [87]:
# Print the orginal labels and predicted labels for the sentence  in test data, which is at index value 10.
idx = 20
print(predict_set.loc[idx, 'sentences'])
print(test_set.loc[idx, 'labels'].split())
print(predict_set.loc[idx, 'labels'])

are there any 24 hour breakfast places nearby 

['O', 'O', 'O', 'B-Hours', 'I-Hours', 'B-Cuisine', 'O', 'B-Location']
['O', 'O', 'O', 'B-Hours', 'I-Hours', 'I-Hours', 'O', 'B-Location']


#Transitions Learned by CRF

In [88]:
from util import print_top_likely_transitions
from util import print_top_unlikely_transitions

In [89]:
print_top_likely_transitions(crf.transition_features_)

B-Restaurant_Name -> I-Restaurant_Name 7.141253
B-Amenity -> I-Amenity 6.807435
B-Location -> I-Location 6.751129
I-Location -> I-Location 6.631297
B-Hours -> I-Hours 6.539660
I-Amenity -> I-Amenity 6.383834
B-Dish -> I-Dish  6.304303
I-Restaurant_Name -> I-Restaurant_Name 6.284943
B-Cuisine -> I-Cuisine 6.073672
I-Hours -> I-Hours 6.024253


In [90]:
print_top_unlikely_transitions(crf.transition_features_)

B-Price -> B-Location -0.776032
I-Location -> B-Dish  -0.852074
I-Price -> B-Location -0.883928
I-Hours -> O       -0.908004
I-Rating -> O       -0.955190
B-Restaurant_Name -> B-Cuisine -0.972640
I-Price -> O       -1.007530
I-Dish -> B-Cuisine -1.040268
I-Restaurant_Name -> B-Dish  -1.201751
I-Restaurant_Name -> B-Cuisine -1.308924
